# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime -> Change runtime type -> T4 GPU, then run this cell.
!nvidia-smi -L
!pip -q install -r requirements-colab.txt


In [ ]:
# Kaggle secrets are optional if archive.zip is uploaded.
import os
try:
    from google.colab import userdata
    ku = userdata.get('KAGGLE_USERNAME')
    kk = userdata.get('KAGGLE_KEY')
    if ku and kk:
        os.environ['KAGGLE_USERNAME'] = ku
        os.environ['KAGGLE_KEY'] = kk
        print('Loaded Kaggle secrets')
except Exception as e:
    print('No Kaggle secrets available; upload archive.zip and use --zip instead:', type(e).__name__)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', type(e).__name__)


## 1. Dữ liệu: tải và chia split

In [ ]:
# Prepare dataset. If archive.zip exists in this repo folder, use it. Otherwise use Kaggle CLI.
from pathlib import Path

if Path('archive.zip').exists():
    !python scripts/prepare_data.py --zip archive.zip
elif Path('/content/archive.zip').exists():
    !cp /content/archive.zip ./archive.zip
    !python scripts/prepare_data.py --zip archive.zip
else:
    !python scripts/prepare_data.py --kaggle


In [ ]:
from pathlib import Path
from src.lab_day10_solution import inspect_splits, visualize_split_examples

summary = inspect_splits('data/splits')
print(summary)
figs = visualize_split_examples('train', 6)
print('Wrote figures:', [str(p) for p in figs])


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import gc
import torch
import src.lab_day10_solution as sol
from src.lab_day10_solution import PROMPT_NORM1000, answer_to_pixel_boxes, objects_to_answer, yolo_to_box
from scripts.labcommon import read_jsonl

# Runtime-safe patch: keep image_grid_thw as [n_images, 3]. This is harmless if the source file is already fixed.
def fixed_encode_example(processor, image, prompt, answer):
    prompt_enc = processor.apply_chat_template(
        sol._message(image, prompt),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    )
    full_enc = processor.apply_chat_template(
        sol._message(image, prompt, answer),
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    )
    n = prompt_enc['input_ids'].shape[1]
    if not torch.equal(full_enc['input_ids'][0, :n], prompt_enc['input_ids'][0]):
        raise ValueError('Prompt tokens are not a prefix of the full conversation.')
    labels = full_enc['input_ids'].clone()
    labels[:, :n] = -100
    full_enc['labels'] = labels
    out = {}
    for k, v in full_enc.items():
        if isinstance(v, torch.Tensor) and k in {'input_ids', 'attention_mask', 'labels', 'mm_token_type_ids'}:
            out[k] = v.squeeze(0)
        else:
            out[k] = v
    return out

sol.encode_example = fixed_encode_example

rows = read_jsonl('data/splits/train.jsonl')
example = rows[0]
answer = objects_to_answer(example['objects'], width=example['width'], height=example['height'])
print(PROMPT_NORM1000)
print(answer)
print(answer_to_pixel_boxes(answer, example['width'], example['height']))


## 3. E0: zero-shot

In [ ]:
# E0 zero-shot. This can take 10-25 minutes on T4.
import gc
import torch
import src.lab_day10_solution as sol

model, processor = sol.load_qwen('Qwen/Qwen3-VL-4B-Instruct')
sol.predict_split(
    model, processor,
    split='test',
    out_path='work/e0.jsonl',
    data_dir='data',
    prompt=sol.PROMPT_NORM1000,
    batch_size=4,
    max_new_tokens=256,
)
del model, processor
gc.collect()
torch.cuda.empty_cache()


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Verify chat template and masked labels on one sample.
import src.lab_day10_solution as sol

model, processor = sol.load_qwen('Qwen/Qwen3-VL-4B-Instruct')
train_ds = sol.RoadDamageDataset('train', data_dir='data')
encoded = sol.encode_example(processor, train_ds[0]['image'], train_ds[0]['prompt'], train_ds[0]['answer'])
label_tokens = encoded['input_ids'][encoded['labels'] != -100]
print('answer tokens:', processor.tokenizer.decode(label_tokens, skip_special_tokens=False))
print('image_grid_thw shape:', encoded['image_grid_thw'].shape, encoded['image_grid_thw'])
del model, processor
gc.collect()
torch.cuda.empty_cache()


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Optional: inspect LoRA config. Training cells below load a fresh model themselves.
from pathlib import Path
import yaml

for cfg in ['configs/e1.yaml', 'configs/e2.yaml', 'configs/e3.yaml', 'configs/e4.yaml']:
    setup = yaml.safe_load(Path(cfg).read_text(encoding='utf-8'))
    print(cfg, setup)


## 6. Vòng huấn luyện

In [ ]:
# Train + infer + score selected experiments.
# For the official lab use 300 steps. If the deadline is close, set MAX_STEPS_OVERRIDE = 100 and state it honestly in REPORT.md.
import gc
import torch
from pathlib import Path
import yaml
import src.lab_day10_solution as sol

EXPERIMENTS = ['e1', 'e2', 'e3', 'e4']  # edit to ['e1'] if you only want one run now
MAX_STEPS_OVERRIDE = None               # set to 100 for an emergency partial run

def override_steps(config_path, steps):
    if steps is None:
        return
    p = Path(config_path)
    d = yaml.safe_load(p.read_text(encoding='utf-8'))
    d['max_steps'] = int(steps)
    p.write_text(yaml.safe_dump(d, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Overrode', config_path, 'max_steps=', steps)

for exp in EXPERIMENTS:
    cfg = f'configs/{exp}.yaml'
    override_steps(cfg, MAX_STEPS_OVERRIDE)
    print('
===== TRAIN', exp.upper(), '=====')
    run = sol.train_experiment(cfg)
    print(run)
    print('
===== INFER/SCORE', exp.upper(), '=====')
    sol.infer_adapter(cfg)
    sol.score_experiment(cfg)
    gc.collect()
    torch.cuda.empty_cache()
    print('Finished', exp.upper())


## 7. Suy luận và chấm E1–E3

In [ ]:
# If you trained experiments manually, run this cell to score any existing predictions.
import src.lab_day10_solution as sol

# sol.score_experiment('configs/e1.yaml')
# sol.score_experiment('configs/e2.yaml')
# sol.score_experiment('configs/e3.yaml')
# sol.score_experiment('configs/e4.yaml')


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E5 reward tests, final report, and submission check.
import src.lab_day10_solution as sol

sol.run_reward_tests()
print('reward tests passed')

!python -m src.report_from_evidence --name "Nguyễn Minh Đức"
!python scripts/check_submission.py
